<a href="https://colab.research.google.com/github/Ziqi-Li/GIS5106/blob/main/notebooks/W12_sam2_text_prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W12 Segmenting remote sensing imagery with text prompts and the Segment Anything Model 2 (SAM 2)

This notebook shows how to generate object masks from text prompts with the Segment Anything Model (SAM).

Make sure you use GPU runtime for this notebook. For Google Colab, go to `Runtime` -> `Change runtime type` and select `GPU` as the hardware accelerator. `CPU` is also fine but will be slow.

## Install dependencies/packages

Run the following cell to install the required dependencies.

In [ ]:
%pip install -U "transformers<4.39.0"

In [ ]:
%pip install segment-geospatial groundingdino-py leafmap localtileserver

In [ ]:
import leafmap

from samgeo.text_sam import LangSAM

## Create an interactive map

centered at FSU.

In [ ]:
m = leafmap.Map(center=[30.44179010929151, -84.2976182150657], zoom=19, height="800px")

m.add_basemap("SATELLITE")

m

## Download a sample image

Pan and zoom the map to select the area of interest. Use the draw tools to draw a polygon or rectangle on the map.

Alternatively, you can also assign a bounding box.

In [ ]:
bbox = m.user_roi_bounds()

#alternatively you can manually specify one:
if bbox is None:
    bbox = [-84.29809675855033, 30.443524831567117, -84.29320586964847, 30.439618789531707]

In [ ]:
image = "Image.tif"# output file for the specified image.

leafmap.map_tiles_to_geotiff(
    output=image, bbox=bbox, zoom=19, source="Satellite", overwrite=True
)

Display the downloaded image on the map.

In [ ]:
m.layers[-1].visible = False

m.add_raster(image, layer_name="Image")
m

## Initialize LangSAM class

The initialization of the LangSAM class might take a minute. The initialization downloads the model weights and sets up the model for inference.

In [ ]:
%pip install segment-geospatial[samgeo2]

In [ ]:
sam = LangSAM(model_type="sam2-hiera-large")

## Specify text prompts

In [ ]:
text_prompt = "tree"

## Segment the image

Part of the model prediction includes setting appropriate thresholds for object detection and text association with the detected objects. These threshold values range from 0 to 1 and are set while calling the predict method of the LangSAM class.

`box_threshold`: This value is used for object detection in the image. A higher value makes the model more selective, identifying only the most confident object instances, leading to fewer overall detections. A lower value, conversely, makes the model more tolerant, leading to increased detections, including potentially less confident ones.

`text_threshold`: This value is used to associate the detected objects with the provided text prompt. A higher value requires a stronger association between the object and the text prompt, leading to more precise but potentially fewer associations. A lower value allows for looser associations, which could increase the number of associations but also introduce less precise matches.

Remember to test different threshold values on your specific data. The optimal threshold can vary depending on the quality and nature of your images, as well as the specificity of your text prompts. Make sure to choose a balance that suits your requirements, whether that's precision or recall.

In [ ]:
sam.predict(image, text_prompt, box_threshold=0.24, text_threshold=0.24)

## Visualize the results

Show the result with bounding boxes on the map.

In [ ]:
sam.show_anns(
    cmap="Greens",
    box_color="red",
    title="Automatic Segmentation of Trees",
    blend=True,
)

Show the result without bounding boxes on the map.

In [ ]:
sam.show_anns(
    cmap="Greens",
    add_boxes=False,
    alpha=0.5,
    title="Automatic Segmentation of Trees",
)

### Export the masks to a GeoDataFrame that can be further converted to shapfile or geojson.

In [ ]:
import geopandas as gpd
from shapely.geometry import shape
from rasterio import features

geoms = []
for mask in sam.masks:
    # Convert mask to float/int if it's boolean
    mask_uint8 = mask.astype("uint8")

    # Trace the shapes
    for geom, val in features.shapes(mask_uint8, mask=mask_uint8):

        geoms.append({"geometry": shape(geom)})

# 3. Create the GeoDataFrame
gdf = gpd.GeoDataFrame(geoms)

# Check the result
print(gdf.head())

In [ ]:
geoms = []
for mask in sam.masks:
    # Convert mask to float/int if it's boolean
    mask_uint8 = mask.astype("uint8")

    # Trace the shapes
    for geom, val in features.shapes(mask_uint8, mask=mask_uint8):

        geoms.append({"geometry": shape(geom)})

# 3. Create the GeoDataFrame
gdf = gpd.GeoDataFrame(geoms)

# Check the result
print(gdf.head())

In [ ]:
gdf.plot()

Similarly we can also try with another keywords "Building"

In [ ]:
sam.predict(image, "building", box_threshold=0.24, text_threshold=0.70)

In [ ]:
sam.show_anns(
    #cmap="Greens",
    box_color="red",
    title="Automatic Segmentation of Buildings",
    #blend=True,

)